# Experimentos Próprios — Datasets Sintéticos com Blender
Este notebook documenta experimentos autorais construídos a partir dos 
conhecimentos adquiridos na reprodução do conteúdo do card (ver repositório 
`synthetic-datasets-blender-fundamentals`). Cada experimento expande o 
script original de Camera Constraints com novas funcionalidades.
## Experimentos realizados
1. Variação automática de cor (randomização)
2. Múltiplos objetos na cena
3. Testes automatizados (pytest)
4. Câmera orbitando em elipse com múltiplos ângulos (mini dataset multi-view)

## Sobre a estrutura do projeto

A lógica de cada experimento foi separada em duas partes:

- **`blender_scripts/experiment_utils.py`** — funções "puras", sem depender do módulo `bpy`
  (geração de cor aleatória, configuração dos objetos, checagem de sobreposição). Essa parte
  roda em qualquer Python normal e é a que testamos com `pytest`.
- **`blender_scripts/render_*.py`** — os scripts que efetivamente rodam dentro do Blender
  (criam a cena, aplicam materiais, renderizam). Esses importam as funções do arquivo acima.

Essa separação foi necessária porque o `bpy` só existe dentro do interpretador embutido do
Blender — não dá pra importar nem testar fora dele.

In [1]:
import sys, os

# permite importar o experiment_utils.py de dentro do notebook
sys.path.append(os.path.abspath("../blender_scripts"))

from experiment_utils import (
    gerar_cor_aleatoria,
    gerar_configuracao_objetos,
    gerar_nome_objeto,
    verificar_sem_sobreposicao,
)

---
## Experimento 1 — Variação automática de cor

**Objetivo:** a cada execução do script, o objeto renderizado recebe uma cor RGB
totalmente aleatória, gerada via `random.random()` para cada canal.

Trecho relevante do script `render_cor_aleatoria.py` (executado dentro do Blender):

```python
mat = bpy.data.materials.new(name="CorAleatoria")
mat.use_nodes = True
principled = mat.node_tree.nodes.get("Principled BSDF")
cor_aleatoria = gerar_cor_aleatoria()
principled.inputs["Base Color"].default_value = cor_aleatoria
target_object.data.materials.append(mat)
```

A geração da cor foi isolada na função `gerar_cor_aleatoria()`, que devolve uma tupla RGBA
com o canal alfa sempre fixo em `1.0`.

In [2]:
for _ in range(3):
    print(gerar_cor_aleatoria())

(0.5315101038331884, 0.4541760293104121, 0.3519448078861803, 1.0)
(0.0798352660679783, 0.12230616209240974, 0.730763085460728, 1.0)
(0.38722731662050525, 0.5100835500972773, 0.7057303652731071, 1.0)


Resultado de 3 renders sucessivos, cada um com uma cor diferente:

![Render cor aleatória 1](../assets/render_cor_aleatoria_1.png)
![Render cor aleatória 2](../assets/render_cor_aleatoria_2.png)
![Render cor aleatória 3](../assets/render_cor_aleatoria_3.png)

**Insight:** como a cor é gerada em runtime e aplicada via nós do material
(Principled BSDF), cada render do dataset sintético carrega uma variação de cor
diferente sem precisar duplicar objetos nem materiais manualmente — isso é
justamente o tipo de randomização que dá robustez a um dataset de treino
(o modelo aprende a reconhecer a forma, não a cor específica de um objeto).

---
## Experimento 2 — Múltiplos objetos na cena

**Objetivo:** em vez de um único objeto, a cena passa a ter três objetos de tipos
diferentes (cubo, esfera, cilindro), cada um em uma posição fixa e com uma cor
própria. A câmera passou a olhar para um `Empty` no centro da cena (em vez de
olhar diretamente para um objeto), permitindo enquadrar todos os objetos ao
mesmo tempo.

Trecho relevante do script `render_multiplos_objetos.py`:

```python
objetos_config = gerar_configuracao_objetos()

for i, config in enumerate(objetos_config):
    if config["tipo"] == "cube":
        bpy.ops.mesh.primitive_cube_add(size=1, location=config["location"])
    elif config["tipo"] == "sphere":
        bpy.ops.mesh.primitive_uv_sphere_add(radius=0.5, location=config["location"])
    elif config["tipo"] == "cylinder":
        bpy.ops.mesh.primitive_cylinder_add(radius=0.5, depth=1, location=config["location"])

    obj = bpy.context.active_object
    obj.name = gerar_nome_objeto(i, config["tipo"])
```

In [3]:
config = gerar_configuracao_objetos()
for i, obj in enumerate(config):
    print(gerar_nome_objeto(i, obj["tipo"]), "->", obj["location"])

print("\nSem sobreposição?", verificar_sem_sobreposicao(config, distancia_minima=1.0))

Object_0_cube -> (-2, 0, 1)
Object_1_sphere -> (0, 0, 1)
Object_2_cylinder -> (2, 0, 1)

Sem sobreposição? True


![Render múltiplos objetos](../assets/render_multiplos_objetos.png)

**Insight:** posicionar objetos manualmente funciona para poucos casos, mas fica
claro que o próximo passo natural (fora do escopo deste card) seria randomizar
também a posição, checando sobreposição automaticamente antes de renderizar —
que é exatamente a função `verificar_sem_sobreposicao()` que já deixamos pronta
e testada no Experimento 3.

---
## Experimento 3 — Testes automatizados (pytest)

**Objetivo:** validar programaticamente o comportamento dos dois experimentos
anteriores, sem depender do Blender estar aberto. Para isso, a lógica de
randomização e configuração foi extraída para `experiment_utils.py` (sem
`import bpy`), permitindo importá-la e testá-la com `pytest` normalmente.

Os testes cobrem:
- Valores de cor sempre dentro do intervalo RGB `[0, 1]`
- Canal alfa sempre fixo em `1.0`
- Reprodutibilidade: mesma seed → mesma cor
- Quantidade e tipos corretos de objetos na cena
- Nomes únicos por objeto
- Detecção de sobreposição entre objetos

32 testes ao todo, organizados em duas classes: `TestCorAleatoria` e
`TestMultiplosObjetos`.

In [4]:
!pytest ../tests/test_experimentos.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /home/maju/synthetic-datasets-blender-experiments/venv/bin/python
cachedir: .pytest_cache
rootdir: /home/maju/synthetic-datasets-blender-experiments
plugins: anyio-4.14.2
collected 32 items                                                             

../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[0] PASSED [  3%]
../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[1] PASSED [  6%]
../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[2] PASSED [  9%]
../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[3] PASSED [ 12%]
../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[4] PASSED [ 15%]
../tests/test_experimentos.py::TestCorAleatoria::test_valores_dentro_do_intervalo_rgb[5]

**Resultado:** 32/32 testes passando.

**Insight:** separar a lógica "de negócio" (geração de cor, configuração de
cena) da parte que efetivamente fala com o `bpy` foi o que tornou os testes
possíveis. Isso também é uma boa prática fora do contexto de datasets
sintéticos: qualquer script que dependa de uma engine externa (Blender, Unity,
etc.) fica muito mais fácil de manter quando as regras puras são isoladas em
funções testáveis, deixando a integração com a engine como uma camada fina por
cima.

---
## Experimento 4 — Câmera orbitando em elipse (múltiplos ângulos)

**Objetivo:** em vez de uma única imagem estática, a câmera passa a percorrer um
caminho elíptico ao redor do objeto, renderizando uma imagem em cada posição.
O resultado é um pequeno dataset multi-view do mesmo objeto, visto de vários
ângulos — o tipo de dado que serve, por exemplo, para treinar modelos que
precisam reconhecer um objeto independente do ponto de vista.

Diferente da técnica original do vídeo (que anima a câmera via keyframes ao
longo da timeline), aqui a posição da câmera no caminho é controlada
diretamente por `offset_factor` dentro de um loop Python — isso elimina a
dependência do frame atual da timeline, que foi a causa do problema de
render preto enfrentado em experimentos anteriores.

Trecho relevante do script `render_camera_elipse.py`:

```python
bpy.ops.curve.primitive_nurbs_circle_add(radius=6, location=(0, 0, 2))
path = bpy.context.active_object
path.scale = (1.0, 0.5, 1.0)   # achata o eixo Y -> vira elipse

follow = camera.constraints.new(type='FOLLOW_PATH')
follow.target = path
follow.use_fixed_location = True   # ignora o frame da timeline

for i in range(N_IMAGENS):
    follow.offset_factor = i / N_IMAGENS
    bpy.context.view_layer.update()
    bpy.ops.render.render(write_still=True)
```

In [6]:
N_IMAGENS = 8
for i in range(N_IMAGENS):
    offset_factor = i / N_IMAGENS
    print(f"Render {i+1}/{N_IMAGENS} -> offset_factor = {offset_factor:.3f}")

Render 1/8 -> offset_factor = 0.000
Render 2/8 -> offset_factor = 0.125
Render 3/8 -> offset_factor = 0.250
Render 4/8 -> offset_factor = 0.375
Render 5/8 -> offset_factor = 0.500
Render 6/8 -> offset_factor = 0.625
Render 7/8 -> offset_factor = 0.750
Render 8/8 -> offset_factor = 0.875


![Elipse 0](../assets/render_elipse_0.png)
![Elipse 2](../assets/render_elipse_2.png)
![Elipse 4](../assets/render_elipse_4.png)
![Elipse 6](../assets/render_elipse_6.png)

**Insight:** controlar `offset_factor` diretamente em Python, em vez de usar
keyframes de animação, se mostrou muito mais confiável para geração de
dataset em lote — o resultado passa a depender só do valor calculado no
loop, sem nenhuma dependência do estado da timeline do Blender (frame atual,
range de animação, etc.), que foi justamente a origem dos problemas de render
preto/errado enfrentados nos primeiros testes com Follow Path.

---
## Conclusão

Os quatro experimentos, juntos, cobrem uma progressão natural: (1) introduzir
variabilidade visual simples (cor), (2) aumentar a complexidade da cena
(múltiplos objetos), (3) garantir que a geração desses dados seja confiável
por meio de testes automatizados, e (4) automatizar a captura de múltiplos
ângulos de um mesmo objeto via controle direto da posição da câmera no
caminho, sem depender de keyframes. Esse conjunto de práticas — randomização
controlada, separação de responsabilidades, testes e controle programático de
câmera — é a base para escalar a criação de datasets sintéticos de poucas
dezenas para milhares de imagens sem perder controle sobre a qualidade e a
consistência dos dados gerados.